In [1]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


PyTorch version: 2.5.1+cu121
CUDA available: True
Device: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
import torch
import torch.nn as nn
import torchvision.models as models

In [3]:
class ResNetMLP(nn.Module):
    def __init__(self, num_classes):
        super(ResNetMLP, self).__init__()
        
        # 1. Load a pretrained ResNet (e.g. ResNet18)
        base_model = models.resnet18(pretrained=True)
        
        # 2. Remove the final FC layer -> get features instead
        self.feature_extractor = nn.Sequential(*list(base_model.children())[:-1])
        
        # 3. MLP classifier head
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),   # ResNet18 outputs 512-dim features
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        features = self.feature_extractor(x)   # shape: (batch, 512, 1, 1)
        features = features.view(features.size(0), -1)  # flatten
        out = self.classifier(features)
        return out


Part A — Jupyter notebook code: sensitivity scan analysis + tiny models

In [1]:
# Cell 1: imports & env check
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import filters, feature, transform, io, color
from scipy import fftpack
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import torch, torchvision
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
import time
import json

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


torch: 2.5.1+cu121
cuda available: True


In [3]:
# Cell 2: locate runs & images
BASE = Path("./mumax_files/logs")
LOG_CSV = BASE / "sensitivity_simulation_log.csv"   # change if needed

# Load the CSV (adjust column names if different)
df_log = pd.read_csv(LOG_CSV)
df_log.head()


,run_id,sample_idx,realization,Dind,Ku1,Aex,alpha,Temp,sigma,mx3_path,ku_map_path,out_dir,returncode,stdout_file,stderr_file,timestamp_start,timestamp_end
0,1,0,0,0.000913,613237.590465,7.412909e-12,0.486974,243.647258,0.095501,mumax_files/logs/run_00001/run_00001.mx3,mumax_files/logs/run_00001/Ku_map.txt,mumax_files/logs/run_00001,1,mumax_files/logs/run_00001/mumax_stdout.txt,mumax_files/logs/run_00001/mumax_stderr.txt,2025-10-05 07:51:56,2025-10-05 07:51:58
1,2,0,1,0.000913,613237.590465,7.412909e-12,0.486974,243.647258,0.095501,mumax_files/logs/run_00002/run_00002.mx3,mumax_files/logs/run_00002/Ku_map.txt,mumax_files/logs/run_00002,1,mumax_files/logs/run_00002/mumax_stdout.txt,mumax_files/logs/run_00002/mumax_stderr.txt,2025-10-05 07:51:58,2025-10-05 07:51:58
2,3,0,2,0.000913,613237.590465,7.412909e-12,0.486974,243.647258,0.095501,mumax_files/logs/run_00003/run_00003.mx3,mumax_files/logs/run_00003/Ku_map.txt,mumax_files/logs/run_00003,1,mumax_files/logs/run_00003/mumax_stdout.txt,mumax_files/logs/run_00003/mumax_stderr.txt,2025-10-05 07:51:58,2025-10-05 07:51:58
3,4,1,0,0.000979,352581.504088,7.740728e-12,0.894504,137.902660,0.175393,mumax_files/logs/run_00004/run_00004.mx3,mumax_files/logs/run_00004/Ku_map.txt,mumax_files/logs/run_00004,1,mumax_files/logs/run_00004/mumax_stdout.txt,mumax_files/logs/run_00004/mumax_stderr.txt,2025-10-05 07:51:58,2025-10-05 07:51:59
4,5,1,1,0.000979,352581.504088,7.740728e-12,0.894504,137.902660,0.175393,mumax_files/logs/run_00005/run_00005.mx3,mumax_files/logs/run_00005/Ku_map.txt,mumax_files/logs/run_00005,1,mumax_files/logs/run_00005/mumax_stdout.txt,mumax_files/logs/run_00005/mumax_stderr.txt,2025-10-05 07:51:59,2025-10-05 07:51:59


In [ ]:
# Cell 3: helper to load an image for a run directory
def load_image_from_run(run_dir: Path):
    """
    Tries to load in order:
      1) final_mz.npy or m_z.npy or image.npy inside run_dir
      2) final.ovf -> requires a converter (not implemented here)
      3) final.png / final.tif
    Returns float32 array (H,W) normalized between 0..1.
    """
    # common npy names
    for nm in ["m_z.npy", "final_mz.npy", "image.npy", "Mz.npy", "m.npy"]:
        p = run_dir / nm
        if p.exists():
            arr = np.load(p).astype(np.float32)
            # ensure 2D
            if arr.ndim == 3 and arr.shape[0] in (1,3):
                arr = arr.squeeze(0) if arr.shape[0] == 1 else np.mean(arr,0)
            # normalize
            arr = (arr - arr.min()) / (arr.ptp() + 1e-12)
            return arr
    # check png/tif
    for ext in ("png","tif","tiff","jpg","jpeg"):
        p = run_dir / f"final.{ext}"
        if p.exists():
            im = io.imread(p)
            if im.ndim==3:
                im = color.rgb2gray(im)
            im = im.astype(np.float32)
            im = (im - im.min()) / (im.ptp() + 1e-12)
            return im
    # Try to find any npy in the folder
    for p in run_dir.glob("*.npy"):
        try:
            arr = np.load(p)
            if arr.ndim == 2:
                arr = arr.astype(np.float32)
                arr = (arr - arr.min()) / (arr.ptp() + 1e-12)
                return arr
        except Exception:
            continue
    # fallback: return None
    return None

# Quick test: load first 10
sample_runs = [Path(r) for r in df_log["out_dir"].values[:10]]
for r in sample_runs:
    arr = load_image_from_run(Path(r))
    print(r, "->", None if arr is None else arr.shape)


In [ ]:
# Cell 4: build dataset arrays (images + target params)
images = []
targets = []
meta = []

for idx, row in df_log.iterrows():
    run_dir = Path(row["out_dir"])
    arr = load_image_from_run(run_dir)
    if arr is None:
        # skip if image not found
        continue
    # downsample to 128x128 for ML (use area average)
    img128 = transform.resize(arr, (128,128), order=1, preserve_range=True, anti_aliasing=True)
    images.append(img128.astype(np.float32))
    # choose target(s) - adapt names to your csv
    # We'll predict Dind and sigma if available; else predict Dind only
    D = float(row.get("Dind", np.nan))
    sigma = float(row.get("sigma", row.get("sigma_rel", np.nan)))
    targets.append([D, sigma])
    meta.append(dict(run_dir=str(run_dir), **row.to_dict()))

images = np.stack(images)  # N, H, W
targets = np.array(targets, dtype=np.float32)  # N, 2
print("Loaded images:", images.shape, "targets:", targets.shape)


In [ ]:
# Cell 5: compute simple image features for each image
def compute_features(img):
    # image: 2D np array 0..1
    feats = {}
    feats["mean"] = float(np.mean(img))
    feats["std"] = float(np.std(img))
    # Sobel/edge density
    edges = filters.sobel(img)
    feats["edge_mean"] = float(edges.mean())
    feats["edge_std"] = float(edges.std())
    feats["edge_frac"] = float((edges > np.percentile(edges, 75)).sum() / edges.size)
    # Otsu threshold segmentation -> domain area fraction
    th = filters.threshold_otsu(img)
    mask = img > th
    feats["domain_fraction"] = float(mask.mean())
    # radial power spectrum peak (estimate typical domain scale)
    # compute 2D FFT power, radial average and find peak freq
    F = fftpack.fftshift(fftpack.fft2(img - img.mean()))
    P = np.abs(F)**2
    # radial average
    cy, cx = np.array(P.shape) // 2
    y, x = np.indices(P.shape)
    r = np.sqrt((x-cx)**2 + (y-cy)**2)
    r = r.astype(np.int32)
    rmax = min(cx, cy)
    radial = np.bincount(r.ravel(), weights=P.ravel())[:rmax]
    radial /= (np.bincount(r.ravel())[:rmax] + 1e-12)
    # find peak radius (skip r=0)
    peak_r = float(np.argmax(radial[1:]) + 1)
    feats["fft_peak_r"] = peak_r
    feats["fft_power_mean"] = float(radial.mean())
    return feats

feat_list = [compute_features(im) for im in images]
df_feats = pd.DataFrame(feat_list)
df_meta = pd.DataFrame(meta)
df_all = pd.concat([df_meta.reset_index(drop=True), df_feats.reset_index(drop=True)], axis=1)
df_all[["mean","std","edge_mean","domain_fraction","fft_peak_r"]].head()


In [ ]:
# Cell 6: quick correlation & RandomForest feature importance
# target: D (index 0); if sigma is NaN you can drop that column
y_D = targets[:,0]
X = df_feats.fillna(0).values

X_train, X_test, y_train, y_test = train_test_split(X, y_D, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=200, random_state=0)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print("RF Dind mse:", mean_squared_error(y_test, y_pred), "r2:", r2_score(y_test,y_pred))

# feature importances
feat_names = list(df_feats.columns)
importances = rf.feature_importances_
for n, imp in sorted(zip(feat_names, importances), key=lambda x:-x[1]):
    print(n, round(imp,4))


In [ ]:
# Cell 7: Tiny CNN baseline (PyTorch) - regression of D only
# Dataset wrapper
class LTEMDataset(Dataset):
    def __init__(self, images, targets, transform=None):
        self.images = images
        self.targets = targets
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = self.images[idx]
        # convert to 1xHxW tensor
        x = torch.from_numpy(img).unsqueeze(0).float()
        # normalize per-image (or use global mean/std)
        x = (x - x.mean()) / (x.std() + 1e-6)
        y = torch.tensor(self.targets[idx,0]).float()  # Dind target
        return x, y

# train/val split
train_idx, val_idx = train_test_split(np.arange(len(images)), test_size=0.2, random_state=0)
train_ds = LTEMDataset(images[train_idx], targets[train_idx])
val_ds = LTEMDataset(images[val_idx], targets[val_idx])
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Tiny CNN model (small)
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 128 -> 64
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 64 -> 32
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1) # -> 1x1
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self,x):
        return self.fc(self.conv(x)).squeeze(-1)

model = TinyCNN().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.SmoothL1Loss()   # Huber-like

# training loop (small)
def train_one_epoch(model, loader, opt, crit, device):
    model.train()
    tot_loss = 0.0
    for x,y in loader:
        x = x.to(device); y = y.to(device)
        opt.zero_grad()
        out = model(x)
        loss = crit(out, y)
        loss.backward()
        opt.step()
        tot_loss += loss.item() * x.shape[0]
    return tot_loss / len(loader.dataset)

def eval_model(model, loader, crit, device):
    model.eval()
    tot_loss = 0.0
    preds, trues = [], []
    with torch.no_grad():
        for x,y in loader:
            x = x.to(device); y = y.to(device)
            out = model(x)
            loss = crit(out, y)
            tot_loss += loss.item() * x.shape[0]
            preds.append(out.cpu().numpy())
            trues.append(y.cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    return tot_loss/len(loader.dataset), preds, trues

# run a few epochs (since dataset small)
n_epochs = 40
best_val = 1e9
for epoch in range(1, n_epochs+1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, preds, trues = eval_model(model, val_loader, criterion, DEVICE)
    t1 = time.time()
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), OUTPUT_BASE / "tinycnn_best.pth")
    if epoch % 5 == 0 or epoch==1:
        print(f"Epoch {epoch}/{n_epochs} train_loss={train_loss:.4e} val_loss={val_loss:.4e} time={t1-t0:.1f}s")

# final metrics
mse = mean_squared_error(trues, preds)
print("Final val mse:", mse, "rmse:", np.sqrt(mse))


Part B — Tentative complete CNN architectures and training recipe

B1 — CNN mirroring the paper (structure & PyTorch class)

In [ ]:
import torch.nn as nn

class PaperCNN(nn.Module):
    def __init__(self, out_dim=1, in_channels=1):
        super().__init__()
        # conv layers with specified filter counts
        filters = [64,64,40,36,32,28,24,20,16,16]
        layers = []
        C = in_channels
        for i, f in enumerate(filters):
            stride = 1 if i < 6 else 2   # first 6 stride=1, last 4 stride=2
            layers.append(nn.Conv2d(C, f, kernel_size=3, stride=stride, padding=1, bias=False))
            layers.append(nn.BatchNorm2d(f))
            layers.append(nn.ReLU(inplace=True))
            C = f
        self.conv = nn.Sequential(*layers)
        # after convs, spatial size reduced. let's adaptively pool to flatten
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        # FC head: paper uses FC1 with 10 units then final output; we add a hidden layer
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(filters[-1], 64),
            nn.ReLU(),
            nn.Linear(64, out_dim)
        )
    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x)
        x = self.fc(x)
        return x.squeeze(-1) if x.shape[-1]==1 else x


B2 — Modern alternative: ResNet18 backbone + small head

In [ ]:
import torchvision.models as models

class ResNetRegressor(nn.Module):
    def __init__(self, out_dim=1, pretrained=True, in_channels=1):
        super().__init__()
        res = models.resnet18(pretrained=pretrained)
        # adapt first conv to accept single-channel
        if in_channels == 1:
            w = res.conv1.weight.data
            res.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            # average weights across RGB channels
            res.conv1.weight.data = w.mean(dim=1, keepdim=True)
        # replace fc
        res.fc = nn.Sequential(
            nn.Linear(res.fc.in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, out_dim)
        )
        self.model = res
    def forward(self, x):
        return self.model(x).squeeze(-1) if self.model.fc[-1].out_features==1 else self.model(x)
